In [1]:
from pyspark.sql import SparkSession

spark=SparkSession.builder \
     .appName("Working with files") \
     .getOrCreate()

In [2]:
from google.colab import files

uploaded=files.upload()

In [7]:
df = spark.read.csv("bookings2.csv", header=True, inferSchema=True)


In [8]:
df.printSchema()

In [9]:
df.count()

In [15]:
from pyspark.sql.functions import col, when, lit, avg, sum

df.filter(col("city").isNull()).show()

In [16]:
df.filter(col("provider").isNull()).show()

In [17]:
df.filter(col("booking_amount").isNull()).show()

In [21]:
df = df.withColumnRenamed('__parsed_extra', 'payment_mode')
df.filter(col('payment_mode').isNull()).show()

In [22]:
for c in df.columns:
   print(c, ":", df.filter(col(c).isNull()).count())


In [23]:
df_drop_all = df.na.drop()
df_drop_all.count()

In [24]:
df_drop_amount = df.na.drop(subset=["booking_amount"])
df_drop_amount.count()

In [25]:
df_drop_imp = df.na.drop(
    subset=["customer_name", "service_type", "booking_amount"]
)
df_drop_imp.count()

In [28]:
df = df.na.fill({"city": "Unknown"})


In [29]:
df = df.na.fill({"provider": "Not Available"})

In [30]:
df = df.na.fill({"payment_mode": "Not Provided"})


In [32]:
df = df.na.fill({"bookin": "Unknown"})


In [33]:
df = df.na.fill({"booking_amount": 0})

In [34]:
df = df.withColumn(
    "data_quality_status",
    when(
        col("customer_name").isNull() |
        col("service_type").isNull() |
        col("booking_amount").isNull(),
        "Incomplete"
    ).otherwise("Complete")
)


In [35]:
df.show(5)

In [36]:
df.groupBy("data_quality_status").count().show()

In [39]:
df = df.withColumn("tax", col("booking_amount") * 0.05)



In [40]:
df = df.withColumn(
    "final_amount",
    col("booking_amount") + col("tax")
)


In [43]:
df.filter(
    col("bookin") == "Confirmed"
).agg(
    sum("booking_amount").alias("total_revenue")
).show()


In [44]:
df.groupBy("service_type") \
  .count() \
  .orderBy("count", ascending=False) \
  .show()


In [49]:
df.write \
  .mode("overwrite") \
  .parquet("clean_bookings.parquet")

print("Clean data saved successfully as clean_bookings.parquet")